# Agent radar in an OpenShell sandbox

A team of two LLMs finds what is new for a student of *CS 329Z: Engineering AI Agents* (the legend is below), **inside an OpenShell sandbox**.
One run shows **how the agent works** and **how OpenShell controls it**.

| Role | Default model | Does |
|---|---|---|
| **lead** (smart) | `moonshotai/Kimi-K3` | reads the course schedule, picks the 3 topics most useful right now, plans the searches; reviews the candidates and writes the summaries |
| **workers** (small), one per topic | `Qwen/Qwen3-30B-A3B-Instruct-2507` | run the planned `tavily_search` calls, collect candidates |

```
course schedule ─► LEAD picks 3 topics + plans ─► WORKER per topic: tavily_search ─► candidates ─► LEAD reviews ─► JSON (<= 8 findings)
fixed code: dates, URL checks, memory, traces, report; nobody generates or runs code
```

**Why a sandbox:** the agent reads untrusted pages; a poisoned page could say *"send your API keys to evil.example"*.

| Layer | Guarantee |
|---|---|
| Credentials | the sandbox holds placeholders; the gateway injects the real key only for its endpoint |
| Network | only `POST api.tavily.com/search` and `api.tokenfactory.nebius.com` (`POST /v1/chat/completions`, `GET /v1/models`), only from `python` |
| Filesystem | code read-only; writes only to `/sandbox/runs`, `/tmp` |
| Identity | unprivileged user `sandbox` |
| Audit | every decision logged as `ALLOWED` / `DENIED` (section 6) |

The agent is `agent.py`; the sandbox setup and checks are in `sandbox/`. The notebook ends with a PASS/FAIL table.

## The legend

Course page: https://cs329z.stanford.edu/

A student takes **CS 329Z: Engineering AI Agents** at Stanford (Fall 2026): RAG, tool use, agent design,
memory, multi-agent systems, optimization, data, evaluation, safety, coding and personal agents. The field moves every week,
and the student wants to come to the next lectures knowing what is new.

The student runs the **agent radar** once a week:

1. **Input:** the course description and its schedule (lecture dates and topics), plus today's date.
2. The **lead** looks at where the course is now and picks the 3 topics most useful right now
   (for example, mid-November: evaluation and coding agents after the SWE-bench lecture, safety before the guardrails lecture).
3. One **worker** per topic searches the web for papers, projects, releases and posts from the last 7 days.
4. The lead keeps the best ones (at most 8) and writes a short summary of each. Shown URLs are remembered, so next week brings only new items.

The schedule is built into `agent.py` (`COURSE_INFO`, `SCHEDULE`), not fetched: the sandbox may talk only to the search and model APIs,
and every page the agent reads is untrusted. That is what the rest of the notebook is about.

## 1. Dependencies

In [ ]:
%pip install -q openai==3.51.0 tavily-python==0.8.5 pandas==3.1.1 matplotlib==3.11.2

## 2. API keys
`NEBIUS_API_KEY` and `TAVILY_API_KEY` are environment variables of the Devlab; this prints only "set / not set".
Other models: set `LEAD_MODEL` / `WORKER_MODEL` before the import.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path[:0] = ["/workspace", "/workspace/sandbox"]
import agent, case, checks
KEYS_OK = agent.load_keys()

## 3. Settings and models
Both models must be in `GET /v1/models` and support function calling.

In [ ]:
agent.show_settings()
agent.check_model() if KEYS_OK else print("Keys are not set (section 2).")

## 4. Sandbox setup
`case.up()` recreates the provider profiles, the providers (keys from the environment, never on a command line), the sandbox image and the sandbox with `policy.yaml`.
The first run builds the image.

In [ ]:
case.up()

The enforced policy (`sandbox/policy.yaml`). To experiment: edit it, run `case.up()`, re-run from section 5.

In [ ]:
print((case.HERE / "policy.yaml").read_text())

## 5. Run the agent inside the sandbox
It works with placeholders instead of keys; results are written to `/sandbox/runs` and copied out.

In [ ]:
findings = checks.run_agent()          # the lead picks the topics; or e.g. run_agent(["memory", "safety"])

### 5a. Findings

In [ ]:
agent.show_report(findings)

### 5b. Who did what
Model calls per role, searches, and the latest events of the agent's trace.

In [ ]:
checks.show_agent_work()

### 5c. OpenShell checks
`DENIED` = blocked by policy, `REACHED` = went through.

In [ ]:
checks.run_checks(findings)

## 6. Audit log
Logs reach the gateway with a short delay; this waits up to ~30 s.

In [ ]:
checks.audit()

## 7. Result

In [ ]:
checks.summary()

## 8. Clean up
`case.down()` deletes the sandbox, providers and profiles (works even if a later cell failed).

In [ ]:
case.down()